<a href="https://colab.research.google.com/github/vaishali27-c/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

One row represents one content page for one client with aggregated search performance metrics.

I will use March 2026 as the development month. The final month (June 2026) will only be used for evaluation.

The goal is to identify which pages should be refreshed first.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- imp_prev30
- visible_queries
- rare_share
- anon_share
- top_query_share

### Label
- is_declining

### Context
- client_hash_id
- content_id
- month

### Excluded
- Future information
- Client names
- URLs

In [3]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [4]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HuggingFace,
    TOKEN '{HF_TOKEN}'
)
""")

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [8]:
for name, src in TABLES.items():
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM {src}")

In [10]:
con.sql("SHOW TABLES").df()

,name
0,dim_clients
1,dim_content
2,fact_daily
3,fact_daily_sample
4,fact_query_90d


In [11]:
df = con.sql("""
SELECT *
FROM fact_daily_sample
LIMIT 5
""").df()

df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1 — Grain

In [12]:
print(df[['report_date', 'client_hash_id', 'content_hash_id']].head())

  report_date           client_hash_id           content_hash_id
0  2026-06-01  client_3ffa76342f366962  content_1a6296faee432dae
1  2026-06-01  client_3ffa76342f366962  content_73f21e612565035a
2  2026-06-01  client_3ffa76342f366962  content_5a5be514ff559598
3  2026-06-01  client_3ffa76342f366962  content_05b377d0c8a5cfd8
4  2026-06-01  client_3ffa76342f366962  content_dc34c661d63e55a9


Query 2 — Row count and date span

In [13]:
print("Rows:", len(df))

print("Date range:")
print(df['report_date'].min())
print(df['report_date'].max())

Rows: 5
Date range:
2026-06-01 00:00:00
2026-06-01 00:00:00


Query 3 — Availability check

In [14]:
con.sql("""
SELECT
COUNT(*) AS total_rows,
COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available
FROM fact_daily_sample
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available
0,11694072,3878937,644726


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset is anonymized and does not contain client names or URLs.

It cannot prove that refreshing content causes ranking improvements.

The analysis supports decision making but does not establish causal relationships.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.